## Output is estimated salary

In [1]:
# Libraries for preprocessing 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# Libraries for model training and performance metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
# Load dataset
data = pd.read_csv("D:/Sanctum/AI-ML-JOURNEY/Deep Learning/Data/Churn_Modelling.csv")

In [3]:
# Droping Irrelevant columns
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [4]:
# Handling gender categorical feature
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])


In [5]:
# Handling Geography categorical column
ohe_geography = OneHotEncoder()
geo_ohe = ohe_geography.fit_transform(data[['Geography']]).toarray() # returns 2d array/df
geo_encoded_df = pd.DataFrame(geo_ohe,columns=ohe_geography.get_feature_names_out(['Geography'])) # columns name
# * combining with the original df
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

In [6]:
# Dividing the data into independent and dependent feature
X = data.drop('EstimatedSalary',axis=1)
y = data['EstimatedSalary']

In [7]:
# train-test split
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.20, random_state=42)
# Scale down 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
# Saving encoder to pickle
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open("ohe_geography.pkl",'wb') as file:
    pickle.dump(ohe_geography,file)
# saving scaler to pickle
with open("scaler.pkl",'wb') as file:
    pickle.dump(scaler,file)

## ANN Regression

In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [10]:
#Building ANN model
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), # HL1 connected to i/p layer
    Dense(32,activation='relu'), # HL2 connected
    Dense(1) # OP regression as activation deault is linear activation function
])

c:\Users\sharm\anaconda3\envs\ai-ml\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
# Compiling the model to perform forward and backward propogation
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Set up the tensorboard
log_dir = "regression_logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [14]:
# Training model
history = model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[tensorflow_callback,early_stopping_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 100377.6016 - mae: 100377.6016 - val_loss: 98520.5078 - val_mae: 98520.5078
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 99639.0469 - mae: 99639.0469 - val_loss: 97044.0078 - val_mae: 97044.0078
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 97062.3516 - mae: 97062.3516 - val_loss: 93294.4844 - val_mae: 93294.4844
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 92012.8281 - mae: 92012.8281 - val_loss: 87041.1797 - val_mae: 87041.1797
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 84606.5703 - mae: 84606.5703 - val_loss: 78853.8984 - val_mae: 78853.8984
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 75762.0000 - mae: 75762.0000 - val_loss: 70089.0078 - val_mae: 70089.0078
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 67106.9453 - mae: 67106.9453 - val_loss: 62456.7031 - val_mae: 62456.7031
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

In [ ]:
# !pip install tensorboard notebook jupyterlab 

In [18]:
# lauch tensorboard extension
%load_ext tensorboard
%tensorboard --logdir regression_logs/fit # bacause of + adding the date time as string

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 14156), started 0:00:55 ago. (Use '!kill 14156' to kill it.)

In [19]:
# Evaluate model on the test data
test_loss, test_mae = model.evaluate(X_test,y_test)
print(test_mae)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 50248.6719 - mae: 50248.6719
50248.671875


In [20]:
model.save('regression.h5')